In [1]:
import warnings
warnings.filterwarnings( 'ignore' )
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import RandomizedSearchCV, GridSearchCV
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.utils import shuffle



In [2]:
data_cleaned = pd.read_csv('../../cleaned_gtd.csv', encoding='ISO-8859-1')

In [3]:
data_cleaned['attack_date'] = pd.to_datetime({'year': data_cleaned['iyear'], 'month': data_cleaned['imonth'], 'day': data_cleaned['iday']})
data_cleaned.sort_values(by=['gname', 'attack_date'], inplace=True)

In [4]:
data_cleaned.head()

,iyear,imonth,iday,extended,country,region,provstate,city,latitude,longitude,...,targtype1,target1,natlty1,gname,individual,weaptype1,nkill,property,ishostkid,attack_date
21219,1989,1,23,0,78,8,Attica,Athens,37.99749,23.762728,...,2,Supreme Court Prosecutor Anastasios vernardos,78.0,1 May,0,5,1.0,0,0.0,1989-01-23
21868,1989,4,10,0,78,8,Attica,Athens,37.99749,23.762728,...,2,"Residence, Prosecution Office Chief Samouil Sa...",78.0,1 May,0,6,0.0,1,0.0,1989-04-10
23587,1989,11,1,0,78,8,Attica,Athens,37.99749,23.762728,...,3,"apt. building, Chief Police General Ioannis An...",78.0,1 May,0,6,0.0,1,0.0,1989-11-01
27844,1991,7,16,0,78,8,Attica,Athens,37.99749,23.762728,...,1,Branch,217.0,1 May,0,6,0.0,1,0.0,1991-07-16
27846,1991,7,16,0,78,8,Attica,Athens,37.99749,23.762728,...,1,Offices,79.0,1 May,0,6,0.0,1,0.0,1991-07-16


In [5]:
data_cleaned = data_cleaned.drop(columns=['attack_date', 'iyear', 'imonth', 'iday'])

In [6]:
data_cleaned.head()

,extended,country,region,provstate,city,latitude,longitude,specificity,vicinity,multiple,...,attacktype1,targtype1,target1,natlty1,gname,individual,weaptype1,nkill,property,ishostkid
21219,0,78,8,Attica,Athens,37.99749,23.762728,1.0,0,0.0,...,1,2,Supreme Court Prosecutor Anastasios vernardos,78.0,1 May,0,5,1.0,0,0.0
21868,0,78,8,Attica,Athens,37.99749,23.762728,1.0,0,0.0,...,1,2,"Residence, Prosecution Office Chief Samouil Sa...",78.0,1 May,0,6,0.0,1,0.0
23587,0,78,8,Attica,Athens,37.99749,23.762728,1.0,0,0.0,...,1,3,"apt. building, Chief Police General Ioannis An...",78.0,1 May,0,6,0.0,1,0.0
27844,0,78,8,Attica,Athens,37.99749,23.762728,1.0,0,0.0,...,3,1,Branch,217.0,1 May,0,6,0.0,1,0.0
27846,0,78,8,Attica,Athens,37.99749,23.762728,1.0,0,0.0,...,3,1,Offices,79.0,1 May,0,6,0.0,1,0.0


In [7]:
data_cleaned.columns

Index(['extended', 'country', 'region', 'provstate', 'city', 'latitude',
       'longitude', 'specificity', 'vicinity', 'multiple', 'success',
       'suicide', 'attacktype1', 'targtype1', 'target1', 'natlty1', 'gname',
       'individual', 'weaptype1', 'nkill', 'property', 'ishostkid'],
      dtype='object')

In [8]:
# creates train and test data, first 70% of each group is added to train and remaining 30% to test
def handle_leakage(df):
    train_frames = []
    test_frames = []

    #first 70% of each groups attacks to training set, remainin 30% to testing set
    for _, group_data in df.groupby('gname'):
        split_point = int(len(group_data) * 0.7)  # 70% for training
        train_frames.append(group_data.iloc[:split_point])
        test_frames.append(group_data.iloc[split_point:])           


    # Concatenate all the group-specific splits into final train and test DataFrames
    train_df = pd.concat(train_frames)
    test_df = pd.concat(test_frames)

    # Shuffle each DataFrame separately
    train_df = shuffle(train_df)
    test_df = shuffle(test_df)

    print(len(train_df))

    return train_df, test_df

In [9]:
sample_sizes = [100, 200, 300, 478]
geodata = ['country', 'region', 'provstate', 'latitude', 'longitude', 'natlty1']

for sample_size in sample_sizes:
    # extract top 30 groups and sample 
    top_30_classes = data_cleaned['gname'].value_counts().head(30).index
    top_30_df = data_cleaned[data_cleaned['gname'].isin(top_30_classes)]
    top_30_df = top_30_df.groupby('gname').sample(n=sample_size, random_state=42)

    features = top_30_df.drop(columns=['gname'])
    labels = top_30_df['gname']

    # greedy integer encoding of features
    for col in features.select_dtypes(include='object').columns:
        if col not in geodata:
            features[col], _ = pd.factorize(features[col])

    features = pd.get_dummies(features, columns=geodata)

    top_30_encoded = pd.concat([features, labels], axis = 1)

    #train test split
    train, test = handle_leakage(top_30_encoded)
    print(train.columns)

    #save to csv
    train.to_csv(f'train{sample_size}_OneHot.csv')
    test.to_csv(f'test{sample_size}_OneHot.csv')




2100
Index(['extended', 'city', 'specificity', 'vicinity', 'multiple', 'success',
       'suicide', 'attacktype1', 'targtype1', 'target1',
       ...
       'natlty1_216.0', 'natlty1_217.0', 'natlty1_222.0', 'natlty1_223.0',
       'natlty1_228.0', 'natlty1_233.0', 'natlty1_238.0', 'natlty1_422.0',
       'natlty1_999.0', 'gname'],
      dtype='object', length=4135)
4200
Index(['extended', 'city', 'specificity', 'vicinity', 'multiple', 'success',
       'suicide', 'attacktype1', 'targtype1', 'target1',
       ...
       'natlty1_222.0', 'natlty1_223.0', 'natlty1_228.0', 'natlty1_233.0',
       'natlty1_238.0', 'natlty1_334.0', 'natlty1_359.0', 'natlty1_422.0',
       'natlty1_999.0', 'gname'],
      dtype='object', length=6958)
6300
Index(['extended', 'city', 'specificity', 'vicinity', 'multiple', 'success',
       'suicide', 'attacktype1', 'targtype1', 'target1',
       ...
       'natlty1_228.0', 'natlty1_230.0', 'natlty1_233.0', 'natlty1_238.0',
       'natlty1_334.0', 'natlty1_359.

KeyboardInterrupt: 